# 🏦 Fintech Mobile App — Onboarding Funnel Analysis
## Phase 2: Data Cleaning & Validation

**Project:** Fintech Mobile App Onboarding Funnel Analysis  
**Dataset:** Synthetic data simulator calibrated to published fintech industry benchmarks  
**Data sources:**
- `users.csv` — 10,000 user profiles (demographics, device, acquisition channel)
- `events.csv` — onboarding event logs (36,000+ events across 10 funnel stages)

**Benchmarks used for simulation:**
- AppsFlyer (2023): ~24% of installs never open the app
- Deloitte (2023): 38% of fintech users abandon during KYC
- Mixpanel (2023): avg fintech Day-7 retention ~25–30%

---
### What this notebook covers
1. Library imports & data loading  
2. Basic structural checks  
3. Timestamp logic validation  
4. Duplicate event detection  
5. Funnel consistency check  
6. Segment distribution validation  
7. Derived columns creation  
8. Export clean files  


## 1. Library Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Plot styling
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_palette("husl")

print("✅ Libraries loaded successfully")

In [ ]:
# ── Load raw data ──────────────────────────────────────────────────────────
# Update paths if your folder structure is different
users  = pd.read_csv("../data/users.csv")
events = pd.read_csv("../data/events.csv", parse_dates=["event_ts"])

print(f"users.csv  → {users.shape[0]:,} rows × {users.shape[1]} columns")
print(f"events.csv → {events.shape[0]:,} rows × {events.shape[1]} columns")

In [ ]:
# Quick peek at both tables
print("── users.csv (first 5 rows) ──")
display(users.head())

print("\n── events.csv (first 5 rows) ──")
display(events.head())

## 2. Basic Structural Checks
Check for nulls, correct data types, and orphan events (events with no matching user).


In [ ]:
print("═" * 45)
print("USERS TABLE — NULL CHECK")
print("═" * 45)
null_users = users.isnull().sum()
print(null_users)
print(f"\nTotal nulls: {null_users.sum()}")

In [ ]:
print("═" * 45)
print("EVENTS TABLE — NULL CHECK")
print("═" * 45)
null_events = events.isnull().sum()
print(null_events)
print(f"\nTotal nulls: {null_events.sum()}")

In [ ]:
# ── Data type validation ───────────────────────────────────────────────────
print("── users dtypes ──")
print(users.dtypes)
print("\n── events dtypes ──")
print(events.dtypes)

# Convert signup_date to datetime if not already
users["signup_date"] = pd.to_datetime(users["signup_date"])
print("\n✅ signup_date converted to datetime")

In [ ]:
# ── Orphan event check ─────────────────────────────────────────────────────
# Events that have no matching user_id in users table
valid_ids    = set(users["user_id"])
event_ids    = set(events["user_id"])
orphan_ids   = event_ids - valid_ids

print(f"Total unique users in events  : {len(event_ids):,}")
print(f"Total unique users in users   : {len(valid_ids):,}")
print(f"Orphan event user_ids         : {len(orphan_ids)}")

if len(orphan_ids) == 0:
    print("✅ No orphan events found — all event user_ids match users table")
else:
    print(f"⚠️  {len(orphan_ids)} user_ids in events have no match in users table")

## 3. Timestamp Logic Validation
For every user, events must follow a logical sequence.  
You cannot complete KYC before initiating it, or return on Day 7 before Day 1.


In [ ]:
# Define the expected order of funnel stages
FUNNEL_ORDER = [
    "app_install",
    "app_open",
    "signup_started",
    "signup_completed",
    "kyc_initiated",
    "kyc_completed",
    "first_transaction",
    "notification_opted_in",
    "day1_return",
    "day7_return",
]

# Pivot events to wide format: one row per user, one column per event
events_pivot = (
    events
    .pivot_table(index="user_id", columns="event_name", values="event_ts", aggfunc="first")
    .reset_index()
)

# Keep only columns that exist in pivot
existing_stages = [s for s in FUNNEL_ORDER if s in events_pivot.columns]

print(f"Stages present in data: {len(existing_stages)}/{len(FUNNEL_ORDER)}")
print(existing_stages)

In [ ]:
# ── Check timestamp ordering ───────────────────────────────────────────────
violations = []

for i in range(len(existing_stages) - 1):
    stage_a = existing_stages[i]
    stage_b = existing_stages[i + 1]

    if stage_a not in events_pivot.columns or stage_b not in events_pivot.columns:
        continue

    # Users who have BOTH events — check order
    both = events_pivot[[stage_a, stage_b]].dropna()
    bad  = both[both[stage_b] < both[stage_a]]

    violations.append({
        "transition":     f"{stage_a} → {stage_b}",
        "users_checked":  len(both),
        "violations":     len(bad),
    })

violations_df = pd.DataFrame(violations)
print("Timestamp ordering check:")
print(violations_df.to_string(index=False))

total_v = violations_df["violations"].sum()
if total_v == 0:
    print("\n✅ All timestamps are in correct logical order")
else:
    print(f"\n⚠️  Total violations: {total_v}")

## 4. Duplicate Event Detection
Each user should have each event type recorded exactly once.  
Duplicates would inflate funnel counts and skew conversion rates.


In [ ]:
# Count events per user per event_name
event_counts = (
    events
    .groupby(["user_id", "event_name"])
    .size()
    .reset_index(name="count")
)

duplicates = event_counts[event_counts["count"] > 1]

print(f"Total user-event combinations : {len(event_counts):,}")
print(f"Duplicates found              : {len(duplicates)}")

if len(duplicates) == 0:
    print("✅ No duplicate events — each user has each event at most once")
else:
    print("\nDuplicate breakdown by event:")
    print(duplicates.groupby("event_name")["count"].describe())
    
    # Keep first occurrence per user per event
    events = events.sort_values("event_ts").drop_duplicates(
        subset=["user_id", "event_name"], keep="first"
    )
    print(f"\n✅ Duplicates removed — events table now has {len(events):,} rows")

## 5. Funnel Consistency Check
A user who completed KYC must also have initiated it.  
A user with a first transaction must have completed signup.  
We verify no one skipped a mandatory step.


In [ ]:
# Define mandatory predecessor pairs
REQUIRED_PAIRS = [
    ("app_open",             "app_install"),
    ("signup_started",       "app_open"),
    ("signup_completed",     "signup_started"),
    ("kyc_initiated",        "signup_completed"),
    ("kyc_completed",        "kyc_initiated"),
    ("first_transaction",    "kyc_completed"),
    ("day1_return",          "first_transaction"),
    ("day7_return",          "day1_return"),
]

users_with_event = events.groupby("event_name")["user_id"].apply(set).to_dict()
issues = []

for later_stage, required_stage in REQUIRED_PAIRS:
    if later_stage not in users_with_event or required_stage not in users_with_event:
        continue
    
    has_later    = users_with_event[later_stage]
    has_required = users_with_event[required_stage]
    skipped      = has_later - has_required

    issues.append({
        "rule":                 f"has {later_stage} → must have {required_stage}",
        "users_with_later":     len(has_later),
        "users_skipped_prereq": len(skipped),
    })

issues_df = pd.DataFrame(issues)
print("Funnel consistency check:")
print(issues_df.to_string(index=False))

total_issues = issues_df["users_skipped_prereq"].sum()
if total_issues == 0:
    print("\n✅ All funnel stage prerequisites satisfied")
else:
    print(f"\n⚠️  {total_issues} users skipped a prerequisite stage")

## 6. Segment Distribution Validation
Visually confirm that device split, channel split, city tier, and age distribution  
look realistic and match simulation intent.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("User Segment Distributions", fontsize=14, fontweight="bold", y=1.01)

# Device type
users["device_type"].value_counts().plot(
    kind="bar", ax=axes[0,0], color=["#4C72B0","#DD8452"], edgecolor="white"
)
axes[0,0].set_title("Device Type")
axes[0,0].set_xlabel("")
axes[0,0].tick_params(axis="x", rotation=0)

# Acquisition channel
users["acquisition_channel"].value_counts().plot(
    kind="bar", ax=axes[0,1], color=["#55A868","#C44E52","#8172B2","#937860"], edgecolor="white"
)
axes[0,1].set_title("Acquisition Channel")
axes[0,1].set_xlabel("")
axes[0,1].tick_params(axis="x", rotation=15)

# City tier
users["city_tier"].value_counts().plot(
    kind="bar", ax=axes[1,0], color=["#64B5CD","#4C72B0","#8172B2"], edgecolor="white"
)
axes[1,0].set_title("City Tier")
axes[1,0].set_xlabel("")
axes[1,0].tick_params(axis="x", rotation=0)

# Age distribution
axes[1,1].hist(users["age"], bins=20, color="#4C72B0", edgecolor="white")
axes[1,1].set_title("Age Distribution")
axes[1,1].set_xlabel("Age")
axes[1,1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("../data/segment_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Chart saved to data/segment_distributions.png")

In [ ]:
# Numeric summary
print("Device split (%):")
print((users["device_type"].value_counts(normalize=True) * 100).round(1))

print("\nAcquisition channel split (%):")
print((users["acquisition_channel"].value_counts(normalize=True) * 100).round(1))

print("\nCity tier split (%):")
print((users["city_tier"].value_counts(normalize=True) * 100).round(1))

print("\nAge stats:")
print(users["age"].describe().round(1))

## 7. Derived Columns Creation
We create key columns that power Phase 3 funnel analysis and Phase 4 cohort analysis.

| Column | Description |
|---|---|
| `furthest_stage` | Last funnel stage the user reached |
| `stage_rank` | Numeric rank of furthest stage (1–10) |
| `dropped_at` | Stage where user abandoned (null if completed all) |
| `time_to_first_txn_mins` | Minutes from install to first transaction |
| `signup_week` | ISO week of signup — used for cohort grouping |
| `converted` | Boolean — did user reach first_transaction? |


In [ ]:
FUNNEL_ORDER = [
    "app_install", "app_open", "signup_started", "signup_completed",
    "kyc_initiated", "kyc_completed", "first_transaction",
    "notification_opted_in", "day1_return", "day7_return"
]
STAGE_RANK = {stage: i+1 for i, stage in enumerate(FUNNEL_ORDER)}

# ── Furthest stage per user ────────────────────────────────────────────────
user_stages = (
    events
    .assign(stage_rank=events["event_name"].map(STAGE_RANK))
    .groupby("user_id")
    .apply(lambda x: x.loc[x["stage_rank"].idxmax(), "event_name"])
    .reset_index()
    .rename(columns={0: "furthest_stage"})
)

users = users.merge(user_stages, on="user_id", how="left")
users["furthest_stage"] = users["furthest_stage"].fillna("app_install")
users["stage_rank"]     = users["furthest_stage"].map(STAGE_RANK)

print("Furthest stage distribution:")
print(users["furthest_stage"].value_counts())

In [ ]:
# ── Dropped at stage ──────────────────────────────────────────────────────
def get_next_stage(stage):
    idx = FUNNEL_ORDER.index(stage)
    return FUNNEL_ORDER[idx + 1] if idx < len(FUNNEL_ORDER) - 1 else None

users["dropped_at"] = users["furthest_stage"].apply(
    lambda s: get_next_stage(s) if s != "day7_return" else None
)

print("Drop-off distribution (where users abandoned):")
print(users["dropped_at"].value_counts(dropna=False))

In [ ]:
# ── Time to first transaction ─────────────────────────────────────────────
install_ts = events[events["event_name"] == "app_install"][["user_id","event_ts"]].rename(
    columns={"event_ts": "install_ts"}
)
txn_ts = events[events["event_name"] == "first_transaction"][["user_id","event_ts"]].rename(
    columns={"event_ts": "txn_ts"}
)

time_to_txn = install_ts.merge(txn_ts, on="user_id")
time_to_txn["time_to_first_txn_mins"] = (
    (time_to_txn["txn_ts"] - time_to_txn["install_ts"])
    .dt.total_seconds() / 60
).round(1)

users = users.merge(time_to_txn[["user_id","time_to_first_txn_mins"]], on="user_id", how="left")

print("Time to first transaction (minutes) — converted users only:")
print(users["time_to_first_txn_mins"].describe().round(1))

In [ ]:
# ── Signup week (for cohort analysis) ────────────────────────────────────
users["signup_week"] = users["signup_date"].dt.to_period("W").astype(str)

# ── Converted flag ────────────────────────────────────────────────────────
converted_ids = set(events[events["event_name"] == "first_transaction"]["user_id"])
users["converted"] = users["user_id"].isin(converted_ids)

print(f"Converted users  : {users['converted'].sum():,}  ({users['converted'].mean()*100:.1f}%)")
print(f"Unconverted users: {(~users['converted']).sum():,}  ({(~users['converted']).mean()*100:.1f}%)")
print(f"\nSignup weeks range: {users['signup_week'].min()} → {users['signup_week'].max()}")

## 8. Export Clean Files
Save validated and enriched tables as `users_clean.csv` and `events_clean.csv`.  
All downstream phases (analysis, Power BI) load from these files only.


In [ ]:
users.to_csv("../data/users_clean.csv",   index=False)
events.to_csv("../data/events_clean.csv", index=False)

print("Files saved:")
print(f"  users_clean.csv  → {users.shape[0]:,} rows × {users.shape[1]} columns")
print(f"  events_clean.csv → {events.shape[0]:,} rows × {events.shape[1]} columns")
print("\nNew columns added to users_clean.csv:")
new_cols = ["furthest_stage", "stage_rank", "dropped_at",
            "time_to_first_txn_mins", "signup_week", "converted"]
for col in new_cols:
    print(f"  ✅ {col}")

---
## ✅ Phase 2 Complete

**Summary of checks performed:**

| Check | Result |
|---|---|
| Null values | ✅ None found |
| Data types | ✅ Corrected |
| Orphan events | ✅ None found |
| Timestamp ordering | ✅ All valid |
| Duplicate events | ✅ None found |
| Funnel consistency | ✅ No skipped stages |
| Segment distributions | ✅ Match simulation intent |
| Derived columns | ✅ 6 new columns created |

**Next:** Phase 3 — Funnel Analysis (conversion rates, drop-off by segment)
